<a href="https://colab.research.google.com/github/karsarobert/LLM2026/blob/main/ChatGpt2025_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chat GPT és más nagy nyelvi modellek alkalmazása
##PTE Gépi tanulás
###6. gyakorlat: JSON feladatok – lépésről lépésre
2026. március 9.

Ez a notebook a **6. gyakorlat** – az 5. gyakorlat JSON feladatainak részletes, lépésről lépésre haladó megoldása, kiegészítve haladó témákkal.

Minden megoldási lépés **külön kódcellában** szerepel, kommenttel ellátva. A cellák sorban futtatva mutatják, hogyan épül fel a megoldás.

**A feladatok:**
1. **7.1** – Webshop rendelés JSON feldolgozása
2. **7.2** – Könyvtári nyilvántartó JSON séma tervezése
3. **7.3** – LLM + JSON pipeline: szállodai foglalás
4. **8.** – Hibás JSON javítása
5. **9.** – Valós REST API feldolgozása
6. **10.** – JSON → DataFrame → Excel pipeline
7. **11.** – Batch LLM feldolgozás
8. **12.** – Prompt vs. séma megbízhatóság mérése
9. **13.** – Séma bővítése iteratív fejlesztéssel

---
## 7.1 Feladat – JSON értelmezése és feldolgozása

Egy webshop rendelési JSON szövegét kell feldolgozni:

1. Mi a vevő neve és városa?
2. Mennyi a tételek összege?
3. Mennyi a végső fizetendő (kedvezménnyel és szállítással)?
4. Hány darab terméket rendelt összesen a vevő?

### 1. lépés – A `json` modul importálása

A Python beépített `json` modulja azonnal elérhető, telepítés nélkül.
Importálás után kiírjuk a megerősítést.

In [ ]:
import json   # beépített JSON kezelő modul – nem kell telepíteni

print('json modul betöltve')

json modul betöltve


### 2. lépés – Az adatok definiálása Python szótárként

A rendelés adatait Python dict-ként adjuk meg, majd `json.dumps()` segítségével
JSON szöveggé alakítjuk – ez szimulálja a valós API választ.

**Python dict → JSON szöveg** irány: `json.dumps()`

In [ ]:
rendeles = {
    'rendeles_id': 'ORD-2024-1042',
    'vevo': {'nev': 'Horváth Mária', 'varos': 'Pécs'},
    'tetelek': [
        {'termek': 'Okosóra Samsung',  'db': 1, 'egysegar': 89000},
        {'termek': 'Töltőkábel USB-C', 'db': 2, 'egysegar':  2500},
        {'termek': 'Védőtok',          'db': 1, 'egysegar':  4500}
    ],
    'szallitasi_dij': 990,
    'kupon': 'PECS10',
    'kedvezmeny_szazalek': 10
}

rendeles_json = json.dumps(rendeles, ensure_ascii=False)  # dict → JSON szöveg
print('JSON létrehozva, hossza:', len(rendeles_json), 'karakter')

JSON létrehozva, hossza: 333 karakter


In [ ]:
rendeles_json

'{"rendeles_id": "ORD-2024-1042", "vevo": {"nev": "Horváth Mária", "varos": "Pécs"}, "tetelek": [{"termek": "Okosóra Samsung", "db": 1, "egysegar": 89000}, {"termek": "Töltőkábel USB-C", "db": 2, "egysegar": 2500}, {"termek": "Védőtok", "db": 1, "egysegar": 4500}], "szallitasi_dij": 990, "kupon": "PECS10", "kedvezmeny_szazalek": 10}'

### 3. lépés – JSON szöveg értelmezése: `json.loads()`

A `json.loads()` a JSON szövegből Python dict-et készít.
Kiírjuk a típust és a legfelső szintű kulcsokat – ezzel ellenőrzük a betöltést.

In [ ]:
adat = json.loads(rendeles_json)  # JSON szöveg → Python dict

print('Típus:', type(adat))          # <class 'dict'>
print('Kulcsok:', list(adat.keys()))

Típus: <class 'dict'>
Kulcsok: ['rendeles_id', 'vevo', 'tetelek', 'szallitasi_dij', 'kupon', 'kedvezmeny_szazalek']


### 4. lépés – A vevő adatainak kinyerése

A `vevo` kulcs értéke egy **beágyazott szótár** (nested dict).
`adat['vevo']` maga is egy dict – ezt nevezzük kétszintű JSON struktúrának.

In [ ]:
vevo = adat['vevo']              # beágyazott dict kinyerése

print('Vevő neve:', vevo['nev'])  # belső kulcs elérése
print('Városa:', vevo['varos'])

Vevő neve: Horváth Mária
Városa: Pécs


In [ ]:
#nyerjük ki a rendeles_id-t


Rendelés azonosítója: ORD-2024-1042


### 5. lépés – A tételek listájának áttekintése

A `tetelek` kulcs értéke egy **lista**: minden elem egy-egy tétel dict.
`for` ciklussal bejárjuk és kiírjuk minden tétel adatait.

In [ ]:
for tetel in adat['tetelek']:           # végigmegyünk a listán
    nev = tetel['termek']               # tétel neve
    db  = tetel['db']                   # darabszám
    ar  = tetel['egysegar']             # egységár
    print(f'  {nev}: {db} db x {ar:,} Ft')

  Okosóra Samsung: 1 db x 89,000 Ft
  Töltőkábel USB-C: 2 db x 2,500 Ft
  Védőtok: 1 db x 4,500 Ft


### 6. lépés – Tételek összegének kiszámítása

Minden tétel értéke: `db × egységár`.
A `sum()` + **generator expression** tömör, hatékony megoldás.

In [ ]:
# sum() + generator: minden tétel db × ár, majd összegzés
osszeg = sum(t['db'] * t['egysegar'] for t in adat['tetelek'])

print(f'Tételek összege: {osszeg:,} Ft')

Tételek összege: 98,500 Ft


In [ ]:
a = [t['db'] * t['egysegar'] for t in adat['tetelek']]

In [ ]:
a

[89000, 5000, 4500]

### 7. lépés – Szállítási díj hozzáadása

A szállítási díj egy numerikus mező – hozzáadjuk a tételek összegéhez.
Ez az alap összeg, **a kedvezmény alkalmazása előtt**.

In [ ]:
szallitas = adat['szallitasi_dij']   # szállítási díj kinyerése
alap = osszeg + szallitas             # alap összeg

print(f'Tételek:   {osszeg:,} Ft')
print(f'Szállítás: {szallitas:,} Ft')
print(f'Alap:      {alap:,} Ft')

Tételek:   98,500 Ft
Szállítás: 990 Ft
Alap:      99,490 Ft


### 8. lépés – Kedvezmény alkalmazása

A kedvezmény %-ban van tárolva. Az arányhoz 100-zal osztunk.
Képlet: `alap × (1 - kedvezmény aránya)` → pl. 10% esetén `alap × 0.90`

In [ ]:
szazalek  = adat['kedvezmeny_szazalek']  # kedvezmény mértéke %-ban
kupon     = adat['kupon']                 # kuponkód
fizetendo = alap * (1 - szazalek / 100)   # végső összeg

print(f'Kupon: {kupon} (-{szazalek}%)')
print(f'Fizetendo: {fizetendo:,.0f} Ft')

Kupon: PECS10 (-10%)
Fizetendo: 89,541 Ft


### 9. lépés – Összes rendelt darab

Ugyanolyan `sum()` + generator módszer, mint az összegnél – most a `db` mezőt adjuk össze.

In [ ]:
osszes_db = sum(t['db'] for t in adat['tetelek'])  # db mezők összege

print(f'Összes rendelt darab: {osszes_db} db')

Összes rendelt darab: 4 db


### Összesített eredmény – 7.1 feladat

Az összes kérdésre választ adtunk. Az előző cellákban kiszámított változókat összegyűjtjük.

In [ ]:
print('Rendelésszám:', adat['rendeles_id'])
print('Vevő:', vevo['nev'], '-', vevo['varos'])
print(f'Tételek összege: {osszeg:,} Ft')
print(f'Fizetendo: {fizetendo:,.0f} Ft  (kupon: {kupon})')
print(f'Darabszám: {osszes_db} db')

Rendelésszám: ORD-2024-1042
Vevő: Horváth Mária - Pécs
Tételek összege: 98,500 Ft
Fizetendo: 89,541 Ft  (kupon: PECS10)
Darabszám: 4 db


---
## 7.2 Feladat – Saját JSON séma tervezése: Könyvtár

Tervezzük meg egy könyvtári nyilvántartó rendszer JSON adatstruktúráját!

**Tárolandó adatok:**
- Könyv: cím, szerzők (lista!), ISBN, kiadási év, műfaj
- Kölcsönzési állapot: szabad-e; ha kölcsönzött: ki, mikor, meddig
- Értékelések: olvasó neve, pontszám (1–5), szöveg, dátum

Egyik könyv **szabad** (értékelés nélkül), másik **kölcsönzött** és **2 értékeléssel**.

### 1. lépés – Az első könyv alap adatai

Először csak az egyszerű mezőket adjuk meg.
A `szerzok` mező **lista** – egy könyvnek több szerzője is lehet.

Az `import json` az előző feladatból már betöltve, de itt is szerepeltetjük.

In [ ]:
import json

konyv1 = {
    'cim':        'A mesterséges intelligencia kora',
    'szerzok':    ['Henry Kissinger', 'Eric Schmidt'],  # lista: több szerző
    'isbn':       '978-963-000-001-1',
    'kiadas_eve': 2022,
    'mufaj':      'ismeretterjesztő'
}

print('Könyv:', konyv1['cim'])
print('Szerzők:', konyv1['szerzok'])

Könyv: A mesterséges intelligencia kora
Szerzők: ['Henry Kissinger', 'Eric Schmidt']


### 2. lépés – Kölcsönzési állapot hozzáadása (szabad könyv)

A `kolcsonzes` mező beágyazott dict. Szabad könyvnél `None` (JSON-ban: `null`) a hiányzó értékek.
A kulcsot **utólag adjuk hozzá** – Python-ban ez teljesen szabályos.

In [ ]:
konyv1['kolcsonzes'] = {    # új kulcs hozzáadása a dict-hez
    'szabad':            True,
    'kolcsonzo':         None,   # senki nem vitte ki
    'kolcsonzes_datuma': None,
    'vissza_datuma':     None
}

print('Szabad:', konyv1['kolcsonzes']['szabad'])

Szabad: True


In [ ]:
#nézzük meg a konyv1 tartalmát



### 3. lépés – Értékelések: üres lista

Az üres lista (`[]`) jelzi, hogy értékelések kerülhetnek ide – `None`-nál jobb tervezési döntés.

In [ ]:
konyv1['ertekelesek'] = []   # üres lista – értékelések helye

print('Első könyv kulcsai:', list(konyv1.keys()))

Első könyv kulcsai: ['cim', 'szerzok', 'isbn', 'kiadas_eve', 'mufaj', 'kolcsonzes', 'ertekelesek']


In [ ]:
#nézzük meg a konyv1 tartalmát



{'cim': 'A mesterséges intelligencia kora',
 'szerzok': ['Henry Kissinger', 'Eric Schmidt'],
 'isbn': '978-963-000-001-1',
 'kiadas_eve': 2022,
 'mufaj': 'ismeretterjesztő',
 'kolcsonzes': {'szabad': True,
  'kolcsonzo': None,
  'kolcsonzes_datuma': None,
  'vissza_datuma': None},
 'ertekelesek': []}

### 4. lépés – A második könyv alap adatai

Egyetlen szerzős könyv – de a `szerzok` **akkor is lista**, egy elemmel.
Ez konzisztens tervezés: az alkalmazás kódja mindig listával számolhat.

In [ ]:
konyv2 = {
    'cim':        'Python mesterfokon',
    'szerzok':    ['Luciano Ramalho'],   # lista – 1 elemmel is lista!
    'isbn':       '978-963-123456-0',
    'kiadas_eve': 2022,
    'mufaj':      'programozás'
}

print('Könyv:', konyv2['cim'])

Könyv: Python mesterfokon


### 5. lépés – Kölcsönzési adat: foglalt állapot

Foglalt könyvnél `szabad: False` és kitöltjük a dátumokat.
ISO 8601 formátum: `ÉÉÉÉ-HH-NN` – ez az univerzálisan elfogadott szabvány.

In [ ]:
konyv2['kolcsonzes'] = {
    'szabad':            False,
    'kolcsonzo':         'Kis Péter',    # ki vitte ki
    'kolcsonzes_datuma': '2025-02-28',   # ISO formátum
    'vissza_datuma':     '2025-03-28'
}

print('Kölcsönző:', konyv2['kolcsonzes']['kolcsonzo'])

Kölcsönző: Kis Péter


### 6. lépés – Értékelések hozzáadása

Két értékelés – mindkettő egy dict a listában.
Minden értékelés tartalmaz: olvasó neve, pontszám, szöveg, dátum.

In [ ]:
konyv2['ertekelesek'] = [
    {'olvaso': 'Nagy Anna',  'pontszam': 5, 'szoveg': 'Kiváló referencia!', 'datum': '2024-11-10'},
    {'olvaso': 'Varga Béla', 'pontszam': 4, 'szoveg': 'Haladóknak ajánlom.','datum': '2025-01-05'}
]

n = len(konyv2['ertekelesek'])   # értékelések száma
print(f'{n} értékelés hozzáadva')

2 értékelés hozzáadva


### 7. lépés – A könyvtár összerakása

Egyetlen dict foglalja össze a könyvtárat. A `konyvek` lista tartalmazza a két könyv dict-et.

In [ ]:
konyvtar = {
    'nev':     'PTE Könyvtár',
    'konyvek': [konyv1, konyv2]   # a két dict bekerül a listába
}

print(f'Könyvtár: {konyvtar["nev"]}, {len(konyvtar["konyvek"])} könyv')

Könyvtár: PTE Könyvtár, 2 könyv


In [ ]:
# Nézzük meg a könyvtár tartalmát

### 8. lépés – Mentés JSON fájlba

`json.dump()` fájlba ír – az `s` nélküli változat.
Fontos paraméterek: `indent=2`, `ensure_ascii=False`, `encoding='utf-8'`.

In [ ]:
with open('konyvtar.json', 'w', encoding='utf-8') as f:
    json.dump(konyvtar, f, indent=2, ensure_ascii=False)  # dict → fájl

print('Fájl mentve: konyvtar.json')

Fájl mentve: konyvtar.json


### 9. lépés – Visszaolvasás és ellenőrzés

`json.load()` fájlból olvas vissza (nincs `s` a végén).
Ellenőrizzük a kölcsönzőt: ez háromszintű elérés.

In [ ]:
with open('konyvtar.json', 'r', encoding='utf-8') as f:
    betoltott = json.load(f)   # fájl → dict

print('Könyvek száma:', len(betoltott['konyvek']))
kolcsonzo = betoltott['konyvek'][1]['kolcsonzes']['kolcsonzo']   # 3 szintes
print('Kölcsönző:', kolcsonzo)

Könyvek száma: 2
Kölcsönző: Kis Péter


In [ ]:
betoltott

{'nev': 'PTE Könyvtár',
 'konyvek': [{'cim': 'A mesterséges intelligencia kora',
   'szerzok': ['Henry Kissinger', 'Eric Schmidt'],
   'isbn': '978-963-000-001-1',
   'kiadas_eve': 2022,
   'mufaj': 'ismeretterjesztő',
   'kolcsonzes': {'szabad': True,
    'kolcsonzo': None,
    'kolcsonzes_datuma': None,
    'vissza_datuma': None},
   'ertekelesek': []},
  {'cim': 'Python mesterfokon',
   'szerzok': ['Luciano Ramalho'],
   'isbn': '978-963-123456-0',
   'kiadas_eve': 2022,
   'mufaj': 'programozás',
   'kolcsonzes': {'szabad': False,
    'kolcsonzo': 'Kis Péter',
    'kolcsonzes_datuma': '2025-02-28',
    'vissza_datuma': '2025-03-28'},
   'ertekelesek': [{'olvaso': 'Nagy Anna',
     'pontszam': 5,
     'szoveg': 'Kiváló referencia!',
     'datum': '2024-11-10'},
    {'olvaso': 'Varga Béla',
     'pontszam': 4,
     'szoveg': 'Haladóknak ajánlom.',
     'datum': '2025-01-05'}]}]}

---
## 7.3 Feladat – LLM + JSON pipeline: Szállodai foglalás

Egy szállodai visszaigazolás e-mail szövegéből kinyerjük az adatokat a Gemini API segítségével.

**Feldolgozandó szöveg:**
> Kedves Kovács János! Örömmel értesítjük, hogy foglalását 2026. július 10–15. között
> rögzítettük. 2 kétágyas szoba, 150 000 Ft. Tartalmazza: reggeli, szaunahasználat, parkoló.

**Séma:** vendég neve, érkezés, távozás, szobák száma, összköltség, szolgáltatások listája

In [ ]:
#!pip install google-generativeai --quiet   # Gemini API csomag

#print('Csomag telepítve.')

### 1. lépés – Modulok importálása

- `google.generativeai` – a Gemini API Python könyvtára
- `GenerationConfig` – JSON kimenet és séma beállításához
- `json` – a válasz szöveg feldolgozásához

In [ ]:
import google.generativeai as genai         # Gemini API
import json                                  # JSON feldolgozás
from google.generativeai.types import GenerationConfig  # konfiguráció

print('Modulok betöltve.')

Modulok betöltve.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### 2. lépés – API kulcs betöltése és modell inicializálása

A Colab `userdata` tárolójából olvassuk a kulcsot.
`try/except` gondoskodik arról, hogy helyi gépen is működjön.

**Fontos:** soha ne írjuk be a kulcsot közvetlenül a kódba!

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')   # Colab tárolóból
except Exception:
    import getpass
    api_key = getpass.getpass('API kulcs: ')   # manuális megadás

genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash')
print('Modell kész: gemini-2.5-flash')

Modell kész: gemini-2.5-flash


### 3. lépés – JSON séma definiálása

A sémában megadjuk a várt struktúrát Python szótárként.
A Gemini API **kötelezően** betartja – garantált a helyes kimenet.

A `szolgaltatasok` mező `array` típusú, elemei `string`-ek.

In [ ]:
foglalas_schema = {
    'type': 'object',
    'properties': {
        'vendeg_neve':    {'type': 'string'},
        'erkezes':        {'type': 'string'},
        'tavozas':        {'type': 'string'},
        'szobak_szama':   {'type': 'integer'},
        'osszkoltseg':    {'type': 'number'},
        'szolgaltatasok': {'type': 'array', 'items': {'type': 'string'}}
    }
}

print('Séma mezői:', list(foglalas_schema['properties'].keys()))

Séma mezői: ['vendeg_neve', 'erkezes', 'tavozas', 'szobak_szama', 'osszkoltseg', 'szolgaltatasok']


In [ ]:
type(foglalas_schema)

dict

In [ ]:
foglalas_schema

{'type': 'object',
 'properties': {'vendeg_neve': {'type': 'string'},
  'erkezes': {'type': 'string'},
  'tavozas': {'type': 'string'},
  'szobak_szama': {'type': 'integer'},
  'osszkoltseg': {'type': 'number'},
  'szolgaltatasok': {'type': 'array', 'items': {'type': 'string'}}}}

### 4. lépés – Az e-mail szövege

Ez lesz az LLM bemenete. Zárójelbe tett string literálokat a Python automatikusan összefűz.
Valós alkalmazásban beérkező e-mailből vagy adatbázisból jönne.

In [ ]:
email_szoveg = (
    'Kedves Kovács János! '
    'Örömmel értesítjük, hogy foglalását '
    '2026. július 10. és 2026. július 15. között rögzítettük. '
    '2 kétágyas szoba, összesen 150 000 Ft. '
    'Tartalmazza: reggeli, szaunahasználat, parkoló. '
    'Hotel Naplemente'
)

print('Email betöltve:', len(email_szoveg), 'karakter')

Email betöltve: 217 karakter


### 5. lépés – Prompt és konfiguráció

Rövid prompt: a séma gondoskodik a pontos struktúráról.
`GenerationConfig` garantálja a JSON kimenetet.

In [ ]:
prompt = 'Nyerd ki az adatokat ebből a szállodai visszaigazolásból:\n\n' + email_szoveg

config = GenerationConfig(
    response_mime_type='application/json',   # JSON kimenet
    response_schema=foglalas_schema          # kötelező struktúra
)

print('Prompt és konfiguráció kész')

Prompt és konfiguráció kész


### 6. lépés – API hívás és válasz feldolgozása

Elküldjük a promptot, majd `json.loads()` segítségével dict-té alakítjuk a választ.
Mivel sémát adtunk meg, garantált a helyes struktúra.

In [ ]:
valasz  = model.generate_content(prompt, generation_config=config)  # API hívás
eredmeny = json.loads(valasz.text)   # JSON szöveg → Python dict

print('Típus:', type(eredmeny))
print('Kulcsok:', list(eredmeny.keys()))

Típus: <class 'dict'>
Kulcsok: ['erkezes', 'osszkoltseg', 'szobak_szama', 'szolgaltatasok', 'tavozas', 'vendeg_neve']


In [ ]:
valasz

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "{\"erkezes\":\"2026. j\u00falius 10.\",\"osszkoltseg\":150000,\"szobak_szama\":2,\"szolgaltatasok\":[\"reggeli\",\"szaunahaszn\u00e1lat\",\"parkol\u00f3\"],\"tavozas\":\"2026. j\u00falius 15.\",\"vendeg_neve\":\"Kov\u00e1cs J\u00e1nos\"}"
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 114,
        "candidates_token_count": 85,
        "total_token_count": 523
      },
      "model_version": "gemini-2.5-flash"
    }),
)

### 7. lépés – Eredmény megjelenítése

Az adatok strukturált dict-ben vannak – könnyen kiírhatók, tárolhatók, továbbíthatók.
A `join()` a lista elemeit vesszővel fűzi össze.

In [ ]:
print('--- Szállodai foglalás adatai ---')
print('Vendég:  ', eredmeny['vendeg_neve'])
print('Érkezés: ', eredmeny['erkezes'])
print('Távozás: ', eredmeny['tavozas'])
print('Szobák:  ', eredmeny['szobak_szama'], 'db')
print('Összeg:  ', eredmeny['osszkoltseg'], 'Ft')
szolg = ', '.join(eredmeny['szolgaltatasok'])   # lista → vesszős szöveg
print('Szolgáltatások:', szolg)

--- Szállodai foglalás adatai ---
Vendég:   Kovács János
Érkezés:  2026. július 10.
Távozás:  2026. július 15.
Szobák:   2 db
Összeg:   150000 Ft
Szolgáltatások: reggeli, szaunahasználat, parkoló


---
## 8. Hibás JSON javítása

Az LLM-ek néha **érvénytelen JSON-t** generálnak. Tipikus hibák:
- Egyes idézőjel (`'`) JSON kulcsoknál (csak dupla (`"`) megengedett)
- Felesleges záró vessző a JSON objektum utolsó eleme után
- Extra szöveg a JSON előtt/után (pl. magyarázó mondat vagy Markdown kódblokk)

Ebben a feladatban megvizsgáljuk ezeket a hibákat és megírjuk az általános javító függvényt.

### 1. lépés – Tipikus hibás JSON szövegek definiálása

Három példát definiálunk – mindhárom olyan hiba, amivel valódi LLM-eknél találkozhatunk.
Az első szándékosan egyes idézőjelet használ, a második felesleges vesszőt tartalmaz,
a harmadik előtte egy magyarázó mondatot tartalmaz.

In [ ]:
import json

hiba1 = "{'nev': 'Anna', 'kor': 25}"           # egyes idézőjel – Python, nem JSON!
hiba2 = '{"nev": "Anna", "kor": 25,}'          # felesleges záró vessző
hiba3 = 'Íme az eredmény: {"nev": "Anna"}'     # szöveg + JSON keveredik

print('3 hibás JSON definiálva')

3 hibás JSON definiálva


### 2. lépés – A hibák azonosítása try/except segítségével

Minden hibás szöveget megpróbálunk betölteni `json.loads()`-zal.
A `json.JSONDecodeError` kivétel tartalmazza a hiba leírását és pozícióját.

Ez mutatja meg, **miért** hibás a JSON – értékes diagnosztikus információ.

In [ ]:
for i, szoveg in enumerate([hiba1, hiba2, hiba3], 1):
    try:
        json.loads(szoveg)            # megpróbáljuk betölteni
        print(f'hiba{i}: OK')
    except json.JSONDecodeError as e:
        print(f'hiba{i}: HIBA – {e.msg}')

hiba1: HIBA – Expecting property name enclosed in double quotes
hiba2: HIBA – Expecting property name enclosed in double quotes
hiba3: HIBA – Expecting value


### 3. lépés – Általános JSON kinyerő függvény

A `json_kinyerese()` függvény három módszert próbál sorban:
1. **Közvetlen parse** – ha az LLM tiszta JSON-t adott
2. **Markdown blokk eltávolítása** – ha ` ```json ... ``` ` jelölők vannak
3. **Nyers karakter keresés** – megkeresi az első `{` és utolsó `}` közötti részt

Ha egyik sem sikerül, `None`-t ad vissza.

In [ ]:
def json_kinyerese(szoveg):
    szoveg = szoveg.strip()              # felesleges whitespace eltávolítása
    try:
        return json.loads(szoveg)        # 1. kísérlet: közvetlen parse
    except json.JSONDecodeError:
        pass
    if '```json' in szoveg:              # 2. kísérlet: Markdown kódblokk
        e = szoveg.find('```json') + 7
        v = szoveg.find('```', e)
        if v != -1:
            try:
                return json.loads(szoveg[e:v].strip())
            except json.JSONDecodeError:
                pass
    e = szoveg.find('{')                 # 3. kísérlet: nyers keresés
    v = szoveg.rfind('}') + 1
    if e != -1 and v > e:
        try:
            return json.loads(szoveg[e:v])
        except json.JSONDecodeError:
            pass
    return None                          # nem sikerült kinyerni

print('Függvény definiálva')

Függvény definiálva


### 4. lépés – A függvény tesztelése

Mindhárom hibás szöveget átadjuk a függvénynek.
A `hiba1` (egyes idézőjel) sajnos javíthatatlan – ez igazi JSON hiba.
A `hiba3` (szöveg + JSON) sikeresen kinyerhető a 3. módszerrel.

In [ ]:
for i, szoveg in enumerate([hiba1, hiba2, hiba3], 1):
    eredmeny = json_kinyerese(szoveg)   # kinyerés megkísérlése
    if eredmeny:
        print(f'hiba{i}: SIKERÜLT → {eredmeny}')
    else:
        print(f'hiba{i}: NEM SIKERÜLT kinyerni')

hiba1: NEM SIKERÜLT kinyerni
hiba2: NEM SIKERÜLT kinyerni
hiba3: SIKERÜLT → {'nev': 'Anna'}


### 5. lépés – Tesztelés Markdown kódblokkal

Az LLM-ek sokszor Markdown kódblokkal adják vissza a JSON-t.
Ellenőrizzük, hogy a függvény ezt is helyesen kezeli-e.

In [ ]:
markdown_valasz = '```json\n{"nev": "Anna", "kor": 25}\n```'  # LLM tipikus formátum

eredmeny = json_kinyerese(markdown_valasz)   # kinyerés
print('Kinyert adat:', eredmeny)             # {'nev': 'Anna', 'kor': 25}

Kinyert adat: {'nev': 'Anna', 'kor': 25}


---
## 9. Valós REST API JSON feldolgozása

A világ tele van **nyilvános, ingyenes REST API-kkal** amelyek JSON-t adnak vissza.
Ezek feldolgozása ugyanolyan, mint az LLM kimeneteké – `requests` + `json.loads()`.

A `frankfurter.app` API valós árfolyamadatokat szolgáltat – **API kulcs nélkül**, ingyenesen.
URL formátum: `https://api.frankfurter.app/latest?from=EUR&to=HUF,USD,GBP`

### 1. lépés – A `requests` modul importálása

A `requests` csomag HTTP kérések küldésére szolgál. Colab-ban előre telepítve van.
Eggyetlen `get()` hívással le tudjuk kérni egy URL tartalmát.

In [ ]:
import requests   # HTTP kérések küldéséhez (Colab-ban előre telepítve)

print('requests betöltve')

requests betöltve


### 2. lépés – API hívás indítása

A `requests.get(url)` GET kérést küld az API-nak.
A válasz státuszkódja megmutatja, sikeres volt-e a kérés: `200` = OK.

In [ ]:
url = 'https://api.frankfurter.app/latest?from=EUR&to=HUF,USD,GBP'
valasz = requests.get(url)          # GET kérés az API felé

print('Státusz:', valasz.status_code)   # 200 = sikeres

Státusz: 200


### 3. lépés – A JSON válasz feldolgozása

A `requests` csomag beépített `.json()` metódusa automatikusan parse-olja a választ.
Ez ekvivalens a `json.loads(valasz.text)` hívással.

Kiírjuk a legfelső szintű kulcsokat és az alap valutát.

In [ ]:
adat = valasz.json()   # JSON válasz → Python dict (automatikus parse)

print('Kulcsok:', list(adat.keys()))
print('Alap valuta:', adat['base'])
print('Dátum:', adat['date'])

Kulcsok: ['amount', 'base', 'date', 'rates']
Alap valuta: EUR
Dátum: 2026-03-06


In [ ]:
adat

{'amount': 1.0,
 'base': 'EUR',
 'date': '2026-03-06',
 'rates': {'GBP': 0.86693, 'HUF': 393.4, 'USD': 1.1561}}

### 4. lépés – Árfolyamok megjelenítése

A `rates` kulcs egy beágyazott dict: valutakód → árfolyam.
A `.items()` metódussal kulcs-érték párokat kapunk a `for` ciklushoz.

In [ ]:
print(f'Árfolyamok ({adat["date"]}):')
for valuta, arfolyam in adat['rates'].items():
    print(f'  1 EUR = {arfolyam:,.2f} {valuta}')

Árfolyamok (2026-03-06):
  1 EUR = 0.87 GBP
  1 EUR = 393.40 HUF
  1 EUR = 1.16 USD


### 5. lépés – Számítás az API adatokkal

Ha tudjuk a HUF árfolyamot, kiszámíthatjuk például a szállodai foglalás euró értékét.
Ez szemlélteti, hogyan épülnek össze a különböző JSON adatforrások egy alkalmazásban.

In [ ]:
huf_arfolyam = adat['rates']['HUF']   # 1 EUR = ? HUF
foglalas_ft  = 150000                  # szállodai foglalás összege (Ft)

eurokban = foglalas_ft / huf_arfolyam  # forintból euróba átváltás
print(f'Foglalás: {foglalas_ft:,} Ft = {eurokban:.2f} EUR')
print(f'(1 EUR = {huf_arfolyam:.2f} HUF, {adat["date"]} árfolyam alapján)')

Foglalás: 150,000 Ft = 381.29 EUR
(1 EUR = 393.40 HUF, 2026-03-06 árfolyam alapján)


---
## 10. JSON → DataFrame → Excel pipeline

A tömeges LLM feldolgozás eredményeit érdemes **Pandas DataFrame-be** tölteni,
amely könnyen exportálható Excelbe – ez egy valós munkahelyi igény.

A pipeline:
```
JSON adatok (list of dicts)
       ↓  pd.DataFrame()
DataFrame (táblázat)
       ↓  df.to_excel()
Excel fájl
```

### 1. lépés – Pandas és openpyxl importálása

`pandas` a táblázatos adatkezelésre, `openpyxl` az Excel fájlok írásához szükséges.
A Colab-ban mindkettő elérhető.

In [ ]:
!pip install openpyxl --quiet   # Excel fájl íráshoz szükséges

import pandas as pd
print('pandas verzió:', pd.__version__)

pandas verzió: 2.2.2


### 2. lépés – LLM elemzési eredmények definiálása

Ezek szimulált LLM kimenetek – mintha 5 vásárlói véleményt elemeztünk volna.
Valódi alkalmazásban ez a Gemini API válaszaiból épülne fel egy `for` ciklusban.

In [ ]:
elemzesek = [
    {'szoveg': 'Remek termék!',         'erzelem': 'pozitív', 'pontszam': 0.95, 'tema': 'termék'},
    {'szoveg': 'Teljesen felesleges.',  'erzelem': 'negatív', 'pontszam': 0.80, 'tema': 'vélemény'},
    {'szoveg': '10 órakor találkozunk.','erzelem': 'semleges','pontszam': 0.10, 'tema': 'időpont'},
    {'szoveg': 'Fantasztikus volt!',    'erzelem': 'pozitív', 'pontszam': 0.98, 'tema': 'esemény'},
    {'szoveg': 'Nem értem a terméket.', 'erzelem': 'negatív', 'pontszam': 0.60, 'tema': 'termék'},
]
print(f'{len(elemzesek)} elemzés definiálva')

5 elemzés definiálva


### 3. lépés – List of dicts → DataFrame

A `pd.DataFrame()` egyetlen sorban táblázattá alakítja a dict listát.
Minden dict egy sor, a kulcsok az oszlopnevek lesznek.

In [ ]:
df = pd.DataFrame(elemzesek)   # lista of dicts → DataFrame (1 sor!)

print(df)

                   szoveg   erzelem  pontszam      tema
0           Remek termék!   pozitív      0.95    termék
1    Teljesen felesleges.   negatív      0.80  vélemény
2  10 órakor találkozunk.  semleges      0.10   időpont
3      Fantasztikus volt!   pozitív      0.98   esemény
4   Nem értem a terméket.   negatív      0.60    termék


### 4. lépés – Gyors statisztikák

A DataFrame lehetővé teszi az azonnali aggregálást.
Szűrés és `groupby()` segítségével látjuk az érzelemenkénti átlagpontszámot.

In [ ]:
print('Átlagos pontszám érzelmenként:')
print(df.groupby('erzelem')['pontszam'].mean().round(2))
print()
print('Pozitív értékelések száma:', len(df[df['erzelem'] == 'pozitív']))

Átlagos pontszám érzelmenként:
erzelem
negatív     0.70
pozitív     0.96
semleges    0.10
Name: pontszam, dtype: float64

Pozitív értékelések száma: 2


### 5. lépés – Exportálás Excel fájlba

A `to_excel()` metódus egy sorban menti a DataFrame-et Excel formátumba.
`index=False` megakadályozza a sor indexek kiírását (szebb táblázat).

In [ ]:
df.to_excel('elemzesek.xlsx', index=False)   # DataFrame → Excel fájl
print('Excel fájl mentve: elemzesek.xlsx')

Excel fájl mentve: elemzesek.xlsx


### 6. lépés – Visszaolvasás és ellenőrzés

`pd.read_excel()` visszatölti a fájlt DataFrame-be.
Ellenőrizzük, hogy a sorok és adatok helyesen tárolódtak.

In [ ]:
df2 = pd.read_excel('elemzesek.xlsx')   # Excel → DataFrame

print('Visszaolvasott sorok:', len(df2))
print(df2[['erzelem', 'pontszam']].head(3))

Visszaolvasott sorok: 5
    erzelem  pontszam
0   pozitív      0.95
1   negatív      0.80
2  semleges      0.10


---
## 11. Batch LLM feldolgozás

Valós alkalmazásokban nem egy, hanem **sok szöveget** kell feldolgozni.
A batch (kötegelt) feldolgozás: egy `for` ciklusban minden szöveget elküldünk az API-nak
és az eredményeket listában gyűjtjük, majd DataFrame-be töltjük.

```
Szövegek listája  →  for ciklus  →  API hívások  →  eredmény lista  →  DataFrame
```

### 1. lépés – API újrainicializálása

Ha ezt a szekciót önállóan futtatják, szükséges az API setup.
A `config` változót is újra definiáljuk a 7.3-as feladatból.

In [ ]:
import google.generativeai as genai
import json
from google.generativeai.types import GenerationConfig

try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    import getpass
    api_key = getpass.getpass('API kulcs: ')

genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash')
print('API kész')

API kész


### 2. lépés – A feldolgozandó foglalások listája

Három különböző szállodai visszaigazolást szimulálunk.
Valódi alkalmazásban ez egy adatbázisból vagy beérkező e-mailek mappájából jönne.

In [ ]:
foglalasok = [
    'Kovács Péter foglalása: 2026.08.10–15., 1 szoba, 75 000 Ft, reggeli.',
    'Nagy Éva foglalása: 2026.09.01–05., 3 szoba, 220 000 Ft, reggeli, parkoló.',
    'Tóth Gábor foglalása: 2026.07.20–22., 2 szoba, 90 000 Ft, szauna.',
]
print(f'{len(foglalasok)} foglalás betöltve')

3 foglalás betöltve


### 3. lépés – A séma és konfiguráció definiálása

Ugyanolyan séma mint a 7.3 feladatban.
Külön cellában szerepel, hogy önállóan is futtatható legyen a szekció.

In [ ]:
foglalas_schema = {
    'type': 'object',
    'properties': {
        'vendeg_neve':    {'type': 'string'},
        'erkezes':        {'type': 'string'},
        'tavozas':        {'type': 'string'},
        'szobak_szama':   {'type': 'integer'},
        'osszkoltseg':    {'type': 'number'},
        'szolgaltatasok': {'type': 'array', 'items': {'type': 'string'}}
    }
}
config = GenerationConfig(
    response_mime_type='application/json',
    response_schema=foglalas_schema
)
print('Séma és konfiguráció kész')

Séma és konfiguráció kész


### 4. lépés – Batch feldolgozás: for ciklus

Minden szöveget elküldünk az API-nak, a dict eredményeket egy listába gyűjtjük.
A `enumerate` megmutatja, hányadik elemet dolgozzuk fel éppen.

In [ ]:
eredmenyek = []                                    # gyűjtőlista

for i, szoveg in enumerate(foglalasok, 1):
    prompt = 'Nyerd ki az adatokat:\n\n' + szoveg
    valasz = model.generate_content(prompt, generation_config=config)
    adat   = json.loads(valasz.text)               # JSON → dict
    eredmenyek.append(adat)                        # lista bővítése
    print(f'{i}. feldolgozva: {adat.get("vendeg_neve", "?")}')

1. feldolgozva: Kovács Péter
2. feldolgozva: Nagy Éva
3. feldolgozva: Tóth Gábor


In [ ]:
eredmenyek

[{'erkezes': '2026.08.10',
  'osszkoltseg': 75000,
  'szobak_szama': 1,
  'szolgaltatasok': ['reggeli'],
  'tavozas': '2026.08.15',
  'vendeg_neve': 'Kovács Péter'},
 {'erkezes': '2026.09.01',
  'osszkoltseg': 220000,
  'szobak_szama': 3,
  'szolgaltatasok': ['reggeli', 'parkoló'],
  'tavozas': '2026.09.05',
  'vendeg_neve': 'Nagy Éva'},
 {'erkezes': '2026.07.20',
  'osszkoltseg': 90000,
  'szobak_szama': 2,
  'szolgaltatasok': ['szauna'],
  'tavozas': '2026.07.22',
  'vendeg_neve': 'Tóth Gábor'}]

### 5. lépés – Eredmények DataFrame-be töltése

Az eredmény lista `pd.DataFrame()`-be kerül – azonnal táblázat!
Ezután aggregálunk: átlagos szobaszám és teljes bevétel.

In [ ]:
import pandas as pd

df_foglalas = pd.DataFrame(eredmenyek)   # list of dicts → DataFrame
print(df_foglalas[['vendeg_neve', 'erkezes', 'szobak_szama', 'osszkoltseg']])
print()
print('Átlagos szobaszám:', df_foglalas['szobak_szama'].mean())
print('Teljes bevétel:',    df_foglalas['osszkoltseg'].sum(), 'Ft')

    vendeg_neve     erkezes  szobak_szama  osszkoltseg
0  Kovács Péter  2026.08.10             1        75000
1      Nagy Éva  2026.09.01             3       220000
2    Tóth Gábor  2026.07.20             2        90000

Átlagos szobaszám: 2.0
Teljes bevétel: 385000 Ft


---
## 12. Prompt vs. séma – megbízhatóság mérése

A két JSON kinyerési módszer összehasonlítása:

| Módszer | Hogyan? | Megbízhatóság |
|---|---|---|
| **Prompt-alapú** | A promptban kérjük a JSON-t | Változó – az LLM néha nem tartja be |
| **Séma-alapú** | `response_schema` az API-ban | Garantált – az API érvényesíti |

Ugyanolyan szöveget küldjük el **3-3-szor** és megszámoljuk a sikeres JSON parse-okat.

### 1. lépés – Prompt-alapú függvény

A prompt utasítja a modellt JSON visszaadására – de nincs API-szintű kényszer.
A függvény visszaadja: `(dict vagy None, siker bool)`.

In [ ]:
def prompt_alapu(szoveg):
    p = f'Adj vissza JSON-t: vendeg_neve, szobak_szama, osszkoltseg.\n\n{szoveg}'
    v = model.generate_content(p)       # séma NÉLKÜLI hívás
    try:
        return json.loads(v.text), True  # (dict, siker=True)
    except json.JSONDecodeError:
        return None, False               # (None, siker=False)

print('Prompt-alapú függvény kész')

Prompt-alapú függvény kész


### 2. lépés – Séma-alapú függvény

Az API `response_schema` paramétere garantálja a struktúrát.
A parse szinte mindig sikerül, ezért a `try/except` itt biztonsági hálóként van.

In [ ]:
def schema_alapu(szoveg):
    p = f'Nyerd ki az adatokat:\n\n{szoveg}'
    v = model.generate_content(p, generation_config=config)  # sémával!
    try:
        return json.loads(v.text), True
    except json.JSONDecodeError:
        return None, False

print('Séma-alapú függvény kész')

Séma-alapú függvény kész


### 3. lépés – Mérés: 3 futtatás mindkét módszerrel

Ugyanazt a szöveget küldjük el 3-szor minden módszerrel.
A `sum()` megszámolja a `True` értékeket (sikeres parse-ok száma).

In [ ]:
teszt = foglalasok[0]   # ugyanaz a szöveg mindkét módszernek
n = 3                   # futtatások száma

prompt_siker = sum(prompt_alapu(teszt)[1] for _ in range(n))
schema_siker = sum(schema_alapu(teszt)[1] for _ in range(n))

print(f'Prompt-alapú: {prompt_siker}/{n} sikeres')
print(f'Séma-alapú:   {schema_siker}/{n} sikeres')

Prompt-alapú: 0/3 sikeres
Séma-alapú:   3/3 sikeres


### 4. lépés – Tartalmi összehasonlítás

Megnézzük, hogy a két módszer **ugyanolyan tartalmú** JSON-t adott-e vissza.
A séma-alapú módszer kulcsai előre ismertek, a prompt-alapú kulcsai véletlenszerűek lehetnek.

In [ ]:
p_adat, _ = prompt_alapu(teszt)   # prompt-alapú eredmény
s_adat, _ = schema_alapu(teszt)   # séma-alapú eredmény

print('Prompt-alapú kulcsok:', list(p_adat.keys()) if p_adat else 'N/A')
print('Séma-alapú kulcsok:  ', list(s_adat.keys()) if s_adat else 'N/A')

Prompt-alapú kulcsok: N/A
Séma-alapú kulcsok:   ['erkezes', 'osszkoltseg', 'szobak_szama', 'szolgaltatasok', 'tavozas', 'vendeg_neve']


---
## 13. Séma bővítése – iteratív fejlesztés

Valós fejlesztés során a JSON séma **fokozatosan bővül** az igények szerint.
Ez az egyik legnagyobb előnye a séma-alapú megközelítésnek: egyszerűen adhatunk új mezőket.

Kiindulás: az eredeti `foglalas_schema` (6 mező).
Bővítés: `foglalas_id` (string) és `ejszakak_szama` (integer) hozzáadása.

### 1. lépés – A bővített séma definiálása

Az eredeti séma `properties` szótárába **két új kulcsot** veszünk fel.
A modell ezeket is kitölti majd a válaszban.

In [ ]:
foglalas_schema_v2 = {
    'type': 'object',
    'properties': {
        'vendeg_neve':    {'type': 'string'},
        'erkezes':        {'type': 'string'},
        'tavozas':        {'type': 'string'},
        'szobak_szama':   {'type': 'integer'},
        'osszkoltseg':    {'type': 'number'},
        'szolgaltatasok': {'type': 'array', 'items': {'type': 'string'}},
        'foglalas_id':    {'type': 'string'},    # ÚJ mező
        'ejszakak_szama': {'type': 'integer'}    # ÚJ mező
    }
}
print('Bővített séma mezői:', list(foglalas_schema_v2['properties'].keys()))

Bővített séma mezői: ['vendeg_neve', 'erkezes', 'tavozas', 'szobak_szama', 'osszkoltseg', 'szolgaltatasok', 'foglalas_id', 'ejszakak_szama']


### 2. lépés – Konfiguráció az új sémával

Egyetlen sor változik az eredeti kódhoz képest: `response_schema=foglalas_schema_v2`.
Ez az iteratív fejlesztés előnye – minimális változtatás, maximális hatás.

In [ ]:
config_v2 = GenerationConfig(
    response_mime_type='application/json',
    response_schema=foglalas_schema_v2   # az új, bővített séma
)

print('Konfiguráció kész a bővített sémával')

Konfiguráció kész a bővített sémával


### 3. lépés – Prompt az új mezőkkel

A promptban megmondjuk a modellnek, hogyan töltse ki az új mezőket.
A `foglalas_id` legyen `FOG-001`, az `ejszakak_szama` az érkezés–távozás különbsége.

In [ ]:
prompt_v2 = (
    'Nyerd ki az adatokat! '
    'A foglalas_id legyen: "FOG-001". '
    'Az ejszakak_szama = tavozas - erkezes napokban.\n\n'
    + email_szoveg   # az eredeti e-mail szöveg (7.3 cellából)
)
print('Prompt kész, hossza:', len(prompt_v2), 'karakter')

Prompt kész, hossza: 321 karakter


### 4. lépés – API hívás a bővített sémával

Elküldjük a promptot az új konfigurációval. A válasz most már **8 mezőt** tartalmaz.

In [ ]:
valasz_v2   = model.generate_content(prompt_v2, generation_config=config_v2)
eredmeny_v2 = json.loads(valasz_v2.text)   # JSON → dict

print(json.dumps(eredmeny_v2, indent=2, ensure_ascii=False))

{
  "ejszakak_szama": 5,
  "erkezes": "2026-07-10",
  "foglalas_id": "FOG-001",
  "osszkoltseg": 150000,
  "szobak_szama": 2,
  "szolgaltatasok": [
    "reggeli",
    "szaunahasználat",
    "parkoló"
  ],
  "tavozas": "2026-07-15",
  "vendeg_neve": "Kovács János"
}


### 5. lépés – Az új mezők ellenőrzése

Ellenőrizzük, hogy a két új mező valóban megjelent-e a válaszban.
A `.get()` biztonságos elérést biztosít – ha a mező hiányzik, `'HIÁNYZIK'` jelenik meg.

In [ ]:
foglalas_id    = eredmeny_v2.get('foglalas_id',    'HIÁNYZIK')   # ÚJ mező
ejszakak_szama = eredmeny_v2.get('ejszakak_szama', 'HIÁNYZIK')   # ÚJ mező

print('Foglalás ID:', foglalas_id)
print('Éjszakák száma:', ejszakak_szama)
print()
print('Összesen', len(eredmeny_v2), 'mező a válaszban (volt: 6, most:', len(eredmeny_v2), ')')

Foglalás ID: FOG-001
Éjszakák száma: 5

Összesen 8 mező a válaszban (volt: 6, most: 8 )


---
## Összefoglalás

| Fejezet | Téma | Kulcstechnika |
|---|---|---|
| 7.1 | JSON betöltés, beágyazott elérés, számítások | `json.loads()`, `sum()` generator |
| 7.2 | Séma tervezés, fájlba mentés, visszaolvasás | `json.dump()`, `json.load()` |
| 7.3 | LLM + séma alapú JSON kimenet | `GenerationConfig`, `response_schema` |
| 8. | Hibás JSON diagnosztika és javítás | `try/except JSONDecodeError`, `find()`, `rfind()` |
| 9. | Valós REST API feldolgozása | `requests.get()`, `.json()` |
| 10. | JSON → DataFrame → Excel export | `pd.DataFrame()`, `to_excel()` |
| 11. | Batch LLM feldolgozás `for` ciklussal | `append()`, `enumerate()`, `pd.DataFrame()` |
| 12. | Prompt vs. séma megbízhatóság mérése | `sum()` + `bool`, összehasonlítás |
| 13. | Séma bővítése iteratív fejlesztéssel | `response_schema` csere, `.get()` |

**Következő óra:** Haladó LLM funkciók – function calling, rendszer promptok, kontextuskezelés.